In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
current_dir = Path.cwd()
df_train_path = str(current_dir.parent / 'datasets/First Level/train_descriptores.csv')
df_valid_path = str(current_dir.parent / 'datasets/First Level/valid_descriptores.csv')
df_test_path = str(current_dir.parent / 'datasets/First Level/test_descriptores.csv')
train_emb_path = str(current_dir.parent / 'datasets/First Level/train_embeddings.npy')
valid_emb_path = str(current_dir.parent / 'datasets/First Level/valid_embeddings.npy')
test_emb_path = str(current_dir.parent / 'datasets/First Level/test_embeddings.npy')
g18_g19_path = str(current_dir.parent / 'datasets/GH18_GH19_descriptores.csv')
g18_g19_emb_path = str(current_dir.parent / 'datasets/GH18_GH19_embeddings.npy')

In [ ]:
train = pd.read_csv(df_train_path)
valid = pd.read_csv(df_valid_path)
test = pd.read_csv(df_test_path)
indexs = [i for i in range(len(train) + len(valid) + len(test))]
combined = pd.concat([train, valid, test], axis=0)
combined['index'] = indexs
combined = combined[combined['ec_number'].str.startswith('3')]
combined = combined[~combined['ec_number'].str.startswith('3.2')]

In [ ]:
from sklearn.model_selection import train_test_split
filtered_index = np.array(combined['index'].values)
no_chitinase_index = filtered_index[:460]
filtered_index = filtered_index[460:]
train_index, test_index = train_test_split(filtered_index, test_size=0.2, random_state=42)

In [ ]:
np.save(str(current_dir.parent / 'datasets/train_index.npy'), train_index)
np.save(str(current_dir.parent / 'datasets/test_index.npy'), test_index)
np.save(str(current_dir.parent / 'datasets/no_chitinase_index.npy'), no_chitinase_index)

In [ ]:
train_index = np.load(str(current_dir.parent / 'datasets/train_index.npy'))
test_index = np.load(str(current_dir.parent / 'datasets/test_index.npy'))
no_chitinase_index = np.load(str(current_dir.parent / 'datasets/no_chitinase_index.npy'))

In [ ]:
no_quitinasas = combined[combined['index'].isin(no_chitinase_index)]
no_quitinasas = no_quitinasas.drop(columns=['index'])
quitinasas = pd.read_csv(g18_g19_path)
quitinasas = quitinasas.iloc[:,1:]
quitinasas = quitinasas.rename(columns={'Secuencia': 'sequence', 'Nombre': 'id_uniprot'})
quitinasas['IsChitinase'] = 1
no_quitinasas['IsChitinase'] = 0
no_quitinasas['Familia'] = 'None'
no_quitinasas.drop(columns=['ec_number', 'IsEnzime'],inplace=True)
test = pd.concat([quitinasas, no_quitinasas], axis=0)
test.drop(columns=['family'], inplace=True)
test.to_csv(str(current_dir.parent / 'datasets/test_vae.csv'), index=False)

In [ ]:
train = combined[combined['index'].isin(train_index)]
train.drop(columns=['index'], inplace=True)
valid = combined[combined['index'].isin(test_index)]
valid.drop(columns=['index'], inplace=True)
train.to_csv(str(current_dir.parent / 'datasets/train_vae.csv'), index=False)
valid.to_csv(str(current_dir.parent / 'datasets/valid_vae.csv'), index=False)

In [ ]:
train_e = np.load(train_emb_path)
valid_e = np.load(valid_emb_path)
test_e = np.load(test_emb_path)
gh18_gh19_e = np.load(g18_g19_emb_path)

combined_e = np.concatenate((train_e, valid_e, test_e), axis=0)
no_quitinasas_e = combined_e[no_chitinase_index]
test_e = np.concatenate((gh18_gh19_e, no_quitinasas_e), axis=0)
np.save(str(current_dir.parent / 'datasets/test_embeddings.npy'), test_e)

train_e = combined_e[train_index]
valid_e = combined_e[test_index]
np.save(str(current_dir.parent / 'datasets/train_embeddings.npy'), train_e)
np.save(str(current_dir.parent / 'datasets/valid_embeddings.npy'), valid_e)